# Ella2027 — AMD MI300X + LTX-2.5

دفتر إعداد مستقل لخادم AMD. لا يغيّر Kaggle ولا فرع GitHub الرئيسي.

يعتمد حاوية AMD الرسمية xDiT، ويُبقي موديل LTX-2.5 محمّلاً في الذاكرة بين المشاهد.

**تنبيه وصول الموديل:** مستودع LTX-2.5 على Hugging Face يتطلب تسجيل الدخول والضغط على Agree and Access مرة واحدة (ليس طلب موافقة على رصيد GPU). بعد ذلك استخدم Hugging Face read token في الخلية التالية.

**لا تشغّل خلية الإطلاق قبل التأكد من الرصيد المجاني والتكلفة.** أول تشغيل ينزّل الأوزان ويهيّئ Torch compile.


In [ ]:
import os, pathlib, subprocess, getpass
print('Python:', subprocess.check_output(['python','--version'], text=True).strip())
print('Docker:', subprocess.run(['docker','--version'], capture_output=True, text=True).stdout.strip() or 'Docker not found')
print('AMD device nodes:')
subprocess.run(['bash','-lc','ls -l /dev/kfd /dev/dri 2>/dev/null || true'])


## 1) تنزيل ملفات العامل من الفرع المنفصل

هذه الخطوة تنزّل الكود فقط؛ لا تنشئ GPU ولا تشغّل توليداً.


In [ ]:
REPO_DIR = pathlib.Path.home() / 'Ella2027-AMD'
if not REPO_DIR.exists():
    subprocess.run(['git','clone','--depth','1','--branch','feature/amd-mi300x-xdit-ltx25-worker','https://github.com/WaelNameh84/Ella2027.git',str(REPO_DIR)], check=True)
else:
    subprocess.run(['git','-C',str(REPO_DIR),'fetch','origin','feature/amd-mi300x-xdit-ltx25-worker'], check=True)
    subprocess.run(['git','-C',str(REPO_DIR),'checkout','feature/amd-mi300x-xdit-ltx25-worker'], check=True)
WORKER_DIR = REPO_DIR / 'services' / 'amd-mi300x-worker'
assert (WORKER_DIR / 'amd_worker.py').is_file()
print('Worker ready:', WORKER_DIR)


## 2) إعداد الاتصال بشكل خاص

أدخل نفس قيمة `KAGGLE_LTX_WORKER_TOKEN` الموجودة في Render/Kaggle، وأدخل Hugging Face read token بعد قبول الوصول للموديل. لا ترسل المفاتيح في المحادثة؛ تحفظ محلياً على خادم AMD فقط ولا تُرفع إلى GitHub.


In [ ]:
api_token = getpass.getpass('Existing Render KAGGLE_LTX_WORKER_TOKEN: ').strip()
if not api_token: raise ValueError('Worker token is required')
hf_token = getpass.getpass('Hugging Face READ token (after Agree and Access): ').strip()
if not hf_token: raise ValueError('Hugging Face token is required for the gated LTX-2.5 repository')
env_lines = [
 'ELLA_API_BASE_URL=https://ella2027.onrender.com',
 'ELLA_AMD_WORKER_TOKEN=' + api_token,
 'HF_TOKEN=' + hf_token,
 'ELLA_WORKER_API_PREFIX=/api/colab-worker',
 'ELLA_XDIT_MODEL=Lightricks/LTX-2.5-Diffusers',
 'ELLA_XDIT_ULYSSES_DEGREE=1',
 'ELLA_XDIT_ATTENTION_BACKEND=aiter',
 'ELLA_XDIT_TORCH_COMPILE=true',
 'ELLA_MAX_SCENE_SECONDS=10', ''
]
(WORKER_DIR / '.env.amd').write_text('\n'.join(env_lines))
os.chmod(WORKER_DIR / '.env.amd', 0o600)
print('Private AMD environment file created; secret is not displayed.')


## 3) فحص قبل التشغيل

لا تنشئ الحاوية ولا تشغّل GPU في هذه الخطوة.


In [ ]:
subprocess.run(['python','-m','py_compile',str(WORKER_DIR / 'amd_worker.py')],check=True)
subprocess.run(['bash','-n',str(WORKER_DIR / 'start_amd_worker.sh')],check=True)
checks = {
 'Docker installed': subprocess.run(['bash','-lc','command -v docker'],capture_output=True,text=True).returncode == 0,
 '/dev/kfd exists': pathlib.Path('/dev/kfd').exists(),
 '/dev/dri exists': pathlib.Path('/dev/dri').exists(),
 'Worker source exists': (WORKER_DIR / 'amd_worker.py').is_file(),
 'Private token file exists': (WORKER_DIR / '.env.amd').is_file(),
}
for name, ok in checks.items(): print(('OK   ' if ok else 'MISSING   ') + name)
if all(checks.values()): print('Preflight passed. No GPU container launched and no credits spent.')
else: print('Preflight incomplete. Do not run the launch cell.')


## 4) تشغيل العامل — يتطلب موافقتك على تكلفة GPU

تشغّل هذه الخلية حاوية AMD الرسمية `rocm/pytorch-xdit:v26.10` على MI300X. سيبدأ تنزيل أوزان LTX-2.5 والتهيئة والـ compile. تكلفة الـ Droplet تستمر طالما الخادم يعمل.


In [ ]:
if not all(checks.values()): raise RuntimeError('Preflight incomplete; refusing to launch.')
subprocess.run(['bash',str(WORKER_DIR / 'start_amd_worker.sh')],check=True)
print('Container launched. Follow startup with: docker logs -f ella-amd-ltx25')
